In [1]:
from sklearn.datasets import fetch_openml

# Adult 数据集：预测收入是否 >50K
adult = fetch_openml("adult", version=2, as_frame=True)
X = adult.data      # 特征 DataFrame
y = adult.target    # 标签 Series（字符串：'>50K'、'<=50K'）

print(X.head())
print(X.dtypes)
print(y.value_counts())

   age  workclass  fnlwgt     education  education-num      marital-status  \
0   25    Private  226802          11th              7       Never-married   
1   38    Private   89814       HS-grad              9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm             12  Married-civ-spouse   
3   44    Private  160323  Some-college             10  Married-civ-spouse   
4   18        NaN  103497  Some-college             10       Never-married   

          occupation relationship   race     sex  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                NaN    Own-child  White  Female             0             0   

   hours-per-week native-country  
0              

In [2]:
import pandas as pd
import numpy as np

# 连续特征：数值型
continuous_features = X.select_dtypes(include=[np.number]).columns.tolist()
# 离散特征：非数值型
discrete_features = [c for c in X.columns if c not in continuous_features]

print("连续特征：", continuous_features)
print("离散特征：", discrete_features)

连续特征： ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
离散特征： ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# 对离散特征做 One-Hot，连续特征原样传递
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), discrete_features),
        ("num", "passthrough", continuous_features),
    ]
)

X_train_sklearn = preprocess.fit_transform(X_train)
X_test_sklearn = preprocess.transform(X_test)

In [5]:
from decision_tree_ID3 import DecisionTreeContinuous as ID3Tree
from decision_tree_C45 import DecisionTreeC45
from decision_tree_CART import DecisionTreeCART
from sklearn.metrics import accuracy_score, classification_report

In [6]:
id3 = ID3Tree(
    min_samples_split=10,
    continuous_features=continuous_features  # 直接用上面识别好的连续特征名列表
)
id3.fit(X_train, y_train)

id3_train_pred = id3.predict(X_train)
id3_test_pred = id3.predict(X_test)

print("ID3 训练集准确率:", accuracy_score(y_train, id3_train_pred))
print("ID3 测试集准确率:", accuracy_score(y_test, id3_test_pred))

ID3 训练集准确率: 0.9353593260990377
ID3 测试集准确率: 0.8198321162901795


In [7]:
c45 = DecisionTreeC45(
    min_samples_split=10,
    min_samples_leaf=5,
    max_depth=None,
    continuous_features=continuous_features
)
c45.fit(X_train, y_train)

c45_train_pred = c45.predict(X_train)
c45_test_pred = c45.predict(X_test)

print("C4.5 训练集准确率:", accuracy_score(y_train, c45_train_pred))
print("C4.5 测试集准确率:", accuracy_score(y_test, c45_test_pred))

C4.5 训练集准确率: 0.9059346573459299
C4.5 测试集准确率: 0.8478809800040947


In [8]:
cart = DecisionTreeCART(
    min_samples_split=10,
    min_samples_leaf=5,
    max_depth=None,
    continuous_features=continuous_features
)
cart.fit(X_train, y_train)

cart_train_pred = cart.predict(X_train)
cart_test_pred = cart.predict(X_test)

print("CART 训练集准确率:", accuracy_score(y_train, cart_train_pred))
print("CART 测试集准确率:", accuracy_score(y_test, cart_test_pred))

CART 训练集准确率: 0.91634736318699
CART 测试集准确率: 0.8354603152937965


In [9]:
from sklearn.tree import DecisionTreeClassifier

# Entropy（类似 ID3/C4.5）
dt_entropy = DecisionTreeClassifier(
    criterion="entropy",
    random_state=42
)
dt_entropy.fit(X_train_sklearn, y_train)

# Gini（类似 CART）
dt_gini = DecisionTreeClassifier(
    criterion="gini",
    random_state=42
)
dt_gini.fit(X_train_sklearn, y_train)

# 评估
dt_entropy_train_pred = dt_entropy.predict(X_train_sklearn)
dt_entropy_test_pred = dt_entropy.predict(X_test_sklearn)

dt_gini_train_pred = dt_gini.predict(X_train_sklearn)
dt_gini_test_pred = dt_gini.predict(X_test_sklearn)

print("sklearn-Entropy 训练集准确率:", accuracy_score(y_train, dt_entropy_train_pred))
print("sklearn-Entropy 测试集准确率:", accuracy_score(y_test, dt_entropy_test_pred))

print("sklearn-Gini 训练集准确率:", accuracy_score(y_train, dt_gini_train_pred))
print("sklearn-Gini 测试集准确率:", accuracy_score(y_test, dt_gini_test_pred))

sklearn-Entropy 训练集准确率: 0.9999415016525783
sklearn-Entropy 测试集准确率: 0.8184672080802566
sklearn-Gini 训练集准确率: 0.9999415016525783
sklearn-Gini 测试集准确率: 0.8217429877840715


In [10]:
import pandas as pd
import matplotlib.pyplot as plt

results = pd.DataFrame({
    "algorithm": ["ID3", "C4.5", "CART", "sklearn-Entropy", "sklearn-Gini"],
    "train_accuracy": [
        accuracy_score(y_train, id3_train_pred),
        accuracy_score(y_train, c45_train_pred),
        accuracy_score(y_train, cart_train_pred),
        accuracy_score(y_train, dt_entropy_train_pred),
        accuracy_score(y_train, dt_gini_train_pred),
    ],
    "test_accuracy": [
        accuracy_score(y_test, id3_test_pred),
        accuracy_score(y_test, c45_test_pred),
        accuracy_score(y_test, cart_test_pred),
        accuracy_score(y_test, dt_entropy_test_pred),
        accuracy_score(y_test, dt_gini_test_pred),
    ],
})

print(results)

         algorithm  train_accuracy  test_accuracy
0              ID3        0.935359       0.819832
1             C4.5        0.905935       0.847881
2             CART        0.916347       0.835460
3  sklearn-Entropy        0.999942       0.818467
4     sklearn-Gini        0.999942       0.821743
